# NB0: Environment-Only Descriptors (POSCAR-based, 99 compounds)

**Goal:** Build a clean pool of structural / environmental descriptors from POSCAR_std
files only (no DOS, no vasprun, no external elemental tables). Then select the best 2-5
by LOO R2 with XGBoost (same pipeline as nb6/nb9), and later merge the winners with the
proven C6 set.

**Why redo this:** the earlier attempt dumped ~100 features onto 99 rows at once, so R2
was bad from overfitting. The fix is not fewer functions, it is: (1) compute neighbours
with periodic images instead of intra-cell distances, (2) normalise per atom, (3) build a
moderate pool, then do feature *selection* instead of throwing everything in.

**This file is cells 1-5 only** (config, load, structure load, geometry analysis). Run it,
send back the output, then we set GRDF/AFS parameters from the real geometry and write the
extraction + model cells.

Location: `Keshav-DDP/environment/nb0_environment_descriptors.ipynb`

## Cell 1: Configuration (paths mirror nb5, no file-guessing)

In [ ]:
import os

# Notebook lives in Keshav-DDP/environment/ (sibling of k-path, new-descriptors, Weight-contribution)
BASE_DIR = os.path.abspath("..")

RASHBA_CSV = os.path.join(BASE_DIR, "Data", "rashba.csv")
VASP_DIR   = os.path.join(BASE_DIR, "Inverse-design", "rashba")   # {Formula}-{uid}/POSCAR_std

RESULTS_DIR = os.path.join(".", "nb0_environment-results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("PATH CHECK")
for name, path in [("BASE_DIR", BASE_DIR), ("RASHBA_CSV", RASHBA_CSV), ("VASP_DIR", VASP_DIR)]:
    flag = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{flag:7s}] {name:11s} = {path}")
print(f"  [OUTPUT ] {'RESULTS_DIR':11s} = {RESULTS_DIR}")

## Cell 2: Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from pymatgen.core.structure import Structure

print("imports OK, pymatgen loaded")

## Cell 3: Load rashba.csv and collapse to 99 compounds

One row per compound, max alpha_R per uid (same target construction as every prior stage).
Columns are printed so we verify names instead of assuming.

In [ ]:
df = pd.read_csv(RASHBA_CSV)
print(f"rashba.csv : {df.shape[0]} rows, {df.shape[1]} cols")
print(f"columns    : {list(df.columns)}")
print(f"unique uids: {df['uid'].nunique()}   unique formulas: {df['Formula'].nunique()}")

TARGET = "Rashba_parameter"  # confirmed from nb5
assert TARGET in df.columns, f"target '{TARGET}' not in columns: {list(df.columns)}"

idx_max = df.groupby("uid")[TARGET].idxmax()
df99 = df.loc[idx_max].reset_index(drop=True)
print(f"\ncollapsed to {len(df99)} compounds (max {TARGET} per uid)")
print(df99[["Formula", "uid", "kpath", TARGET]].head(8).to_string(index=False))
print(f"\n{TARGET}: min={df99[TARGET].min():.3f}  max={df99[TARGET].max():.3f}  mean={df99[TARGET].mean():.3f}")

## Cell 4: Load structures from POSCAR_std

`find_compound_folder` and `load_structure` are taken verbatim from nb5. The `max_Z<=3`
guard catches the pymatgen element-misidentification bug flagged in the progress notes
(it should not fire on POSCAR_std, but we check).

In [ ]:
def find_compound_folder(uid, vasp_dir):
    """From nb5. Folder named {Formula}-{uid}; match the part after the last '-'."""
    for folder in glob.glob(os.path.join(vasp_dir, "*")):
        folder_name = os.path.basename(folder)
        idx = folder_name.rfind("-")
        if idx != -1 and folder_name[idx + 1:] == uid:
            return folder
    return None


def load_structure(compound_folder):
    """From nb5. POSCAR_std first, then encoded path, then plain POSCAR."""
    std_path = os.path.join(compound_folder, "POSCAR_std")
    if os.path.exists(std_path):
        return Structure.from_file(std_path)
    matches = glob.glob(os.path.join(compound_folder, "ss_2d*POSCAR"))
    if matches:
        return Structure.from_file(matches[0])
    direct = os.path.join(compound_folder, "POSCAR")
    if os.path.exists(direct):
        return Structure.from_file(direct)
    return None


structures = {}
missing, bad_elem = [], []
for uid in df99["uid"]:
    folder = find_compound_folder(uid, VASP_DIR)
    if folder is None:
        missing.append(uid); continue
    s = load_structure(folder)
    if s is None:
        missing.append(uid); continue
    if max(site.specie.Z for site in s) <= 3:
        bad_elem.append(uid)
    structures[uid] = s

print(f"loaded {len(structures)} / {len(df99)} structures")
print(f"missing           : {len(missing)}  {missing[:5]}")
print(f"max_Z<=3 (suspect): {len(bad_elem)}  {bad_elem[:5]}")

## Cell 5: Geometry analysis (this is the part to read carefully)

Pure diagnostics, no descriptors computed yet. For each compound it reports atom count,
lattice parameters and gamma, which axis is vacuum, slab thickness, vacuum gap, in-plane
lattice range, nearest bond, and average neighbour count (with periodic images) at several
cutoffs. The cutoff/center/sigma choices for GRDF and AFS come from this output.

In [ ]:
CUTOFFS = (4, 5, 6, 7, 8)

def analyze_geometry(s, cutoffs=CUTOFFS):
    cart = np.array([site.coords for site in s])
    L = np.array(s.lattice.abc)
    _, _, gamma = s.lattice.angles
    ext = cart.max(axis=0) - cart.min(axis=0)      # cartesian spread of atoms per axis
    vac_axis = int(np.argmax(L))                   # C2DB convention: c (z) is vacuum
    slab = ext[vac_axis]
    vacuum = L[vac_axis] - slab
    in_plane = [L[i] for i in range(3) if i != vac_axis]

    n = len(s)
    nbrs = s.get_all_neighbors(max(cutoffs))       # periodic images included
    dists = []
    for site_nbrs in nbrs:
        for nb in site_nbrs:
            d = getattr(nb, "nn_distance", None)
            if d is None:
                d = nb[1]
            dists.append(d)
    dists = np.array(dists)
    counts = {rc: float((dists <= rc).sum()) / n for rc in cutoffs}
    min_bond = float(dists[dists > 0.4].min()) if dists.size else np.nan

    out = dict(n_atoms=n, a=round(L[0], 3), b=round(L[1], 3), c=round(L[2], 3),
               gamma=round(gamma, 1), vac_axis=vac_axis,
               slab_thick=round(slab, 3), vacuum=round(vacuum, 3),
               inplane_min=round(min(in_plane), 3), inplane_max=round(max(in_plane), 3),
               min_bond=round(min_bond, 3))
    for rc in cutoffs:
        out[f"avg_nbr_{rc}A"] = round(counts[rc], 2)
    return out


rows = []
for uid, s in structures.items():
    f = df99.loc[df99["uid"] == uid, "Formula"].iloc[0]
    d = analyze_geometry(s); d["Formula"] = f; d["uid"] = uid
    rows.append(d)

geo = pd.DataFrame(rows)
cols = ["Formula", "uid", "n_atoms", "a", "b", "c", "gamma", "vac_axis",
        "slab_thick", "vacuum", "inplane_min", "inplane_max", "min_bond"] + \
       [f"avg_nbr_{rc}A" for rc in CUTOFFS]
geo = geo[cols]

print("PER-COMPOUND (first 12):")
print(geo.head(12).to_string(index=False))

print("\nSUMMARY (numeric describe):")
print(geo.describe().T[["min", "25%", "50%", "75%", "max"]].round(2).to_string())

print("\nvacuum-axis counts:", geo["vac_axis"].value_counts().to_dict())
print("atom-count counts  :", geo["n_atoms"].value_counts().to_dict())
print("gamma values       :", sorted(geo["gamma"].round(0).unique().tolist()))

geo.to_csv(os.path.join(RESULTS_DIR, "geometry_analysis.csv"), index=False)
print("\nsaved -> nb0_environment-results/geometry_analysis.csv")